<a href="https://colab.research.google.com/github/Deepshika-Mekala/deepshikamekala.github.io/blob/main/ENEL_680_Applied_Optimization_for_Sustainable_Design_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tittle:Sustainable Disposal and Donation Optimization for Fast Fashion Retailers




Deepshika Mekala (30186649)




In [ ]:
#imports
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ModuleNotFoundError: No module named 'gurobipy'

In [ ]:
# Load your dataset
df = pd.read_csv('final_dataset1.csv')
df

In [ ]:
# Rename columns to meaningful names
column_mappings = {
    'Item_Type': 'I',
    'Item_Value': 'Vi',
    'Processing_Cost': 'Ci',
    'Environmental_Impact': 'Ei',
    'Processing_Partner': 'S',
    'Processing_Success_Rate': 'Ps',
    'Donation_Success_Rate': 'Ds',
    'Budget_Per_Item': 'B',
    'Item_Quantity': 'Qi',
    'Resale_Value': 'Ri',
    'Processing_Time': 'Ti'
}
df.rename(columns=column_mappings, inplace=True)



## System Model and Problem Formulation

In this section, we define the system model and formulate the optimization problem for sustainable waste management in the fast fashion industry.

### Problem Formulation

The goal is to optimize the distribution of unsold inventory to recycling facilities and donation centers, minimizing the overall environmental impact and costs. The optimization is subjected to constraints like transportation limits, facility capacities, and environmental regulations.

In [ ]:
# Create a new model
model = gp.Model("sustainable_fashion_optimization")

# Sets
I = df['I'].unique() # Items
S = df['S'].unique() # Partners

In [ ]:
# Parameters from the dataset
Vi = dict(zip(df['I'], df['Vi']))  # Value per item
Ei = dict(zip(df['I'], df['Ei']))  # Environmental impact per item
Ps = dict(zip(df['S'], df['Ps']))  # Processing success rate
Ds = dict(zip(df['S'], df['Ds']))  # Donation success rate
Qi = dict(zip(df['I'], df['Qi']))  # Quantity per item
B = dict(zip(df['I'], df['B']))    # Budget per item
Ri = dict(zip(df['I'], df['Ri']))  # Resale value per item
Ti = dict(zip(df['I'], df['Ti']))  # Time to process each item

# Parameters
cost_dispose = 1
cost_upcycle = 2
cost_recycle = 1.5
cost_donate = 1
budget = 10000  # Total budget
value_per_item = df.set_index('I')['Vi'].to_dict()  # Value per item
env_impact_per_item = df.set_index('I')['Ei'].to_dict()  # Environmental impact per item

In [ ]:
# Decision variables
Q_dispose = model.addVars(I, name="Q_dispose")
Q_upcycle = model.addVars(I, name="Q_upcycle")
Q_recycle = model.addVars(I, name="Q_recycle")
Q_donate = model.addVars(I, S, name="Q_donate")

value_upcycle_multiplier = 1.2  # Adjust this multiplier to incentivize upcycling
value_recycle_multiplier = 1.1  # Adjust this multiplier to incentivize recycling

#### Objective Function:

**Maximize**:
- Minimise Environmental and Social Impact.

$$
    R = \sum_{i \in I} E_i(Q_{\text{dispose}} + Q_{\text{recycle}} + Q_{\text{upcycle}}) + \sum_{s \in S} D_s P_s Q_{\text{donate}}
$$


 **secondary objective function:**
$$
  \text{s} = \sum_{i \in I} \left( V_i[i] \times \left( \text{value\_upcycle\_multiplier} \times Q_{\text{upcycle}}[i] + \text{value\_recycle\_multiplier} \times Q_{\text{recycle}}[i] + \sum_{\text{all}} Q_{\text{donate}}[i, *] \right) \right)
$$

**Enivronmental impact:**
$$
  \text{E} = \sum_{i \in I} \left( E_i[i] \times \left( Q_{\text{dispose}}[i] + Q_{\text{upcycle}}[i] + Q_{\text{recycle}}[i] \right) \right)
$$

In [ ]:
#Objective Function
primaryobjective = gp.quicksum(value_per_item[i] * (value_upcycle_multiplier * Q_upcycle[i] +
                                             value_recycle_multiplier * Q_recycle[i] +
                                             Q_donate.sum(i, '*')) for i in I)
#environmental impact
secondaryobjective = gp.quicksum(env_impact_per_item[i] * (Q_dispose[i] + Q_upcycle[i] + Q_recycle[i]) for i in I)

#Set objective function priorities
model.setObjectiveN(primaryobjective, 0, 1)
model.setObjectiveN(secondaryobjective, 1, 2)

model.ModelSense = GRB.MAXIMIZE

#### Constraints:

**Allocation of donations**:
$$
\sum_{s \in S} P_s Q_{\text{donate}} \leq \sum_{i \in I} Q_i
$$

**Budget allocation**:
$$
 \sum_{i \in I} V_i(Q_{\text{dispose}} + Q_{\text{recycle}} + Q_{\text{upcycle}} + Q_{\text{donate}}) \leq B
$$

**Waste Reduction**:
$$
 Q_{\text{dispose}} + Q_{\text{recycle}} + Q_{\text{upcycle}} + Q_{\text{donate}} \leq \sum_{i \in I} Q_i \
$$

**Min Quota**:
$$
Q_{\text{recycle}} \geq 0.10 \times \sum_{i \in I} Q_i \
$$

$$
Q_{\text{upcycle}} \geq 0.10 \times \sum_{i \in I} Q_i \
$$


In [ ]:
# Non-negativity constraints
model.addConstrs(Q_dispose[i] >= 0 for i in I)
model.addConstrs(Q_upcycle[i] >= 0 for i in I)
model.addConstrs(Q_recycle[i] >= 0 for i in I)
model.addConstrs(Q_donate[i, s] >= 0 for i in I for s in S)

# Budget constraint
total_cost = gp.quicksum( cost_dispose * Q_dispose[i] + cost_upcycle * Q_upcycle[i] +
                         cost_recycle * Q_recycle[i] + cost_donate * Q_donate[i, s]
                         for i in I for s in S)
model.addConstr(total_cost <= budget, "Budget")

# Donation constraint
donation_cost = gp.quicksum(Ps[s] * Q_donate.sum('*', s) for s in S)
model.addConstr(donation_cost <= gp.quicksum(Qi[i] for i in I), name="Donations")

#Waste constraint
waste_cost = gp.quicksum(Q_dispose[i] + Q_recycle[i] + Q_upcycle[i] + gp.quicksum(Q_donate[i, s] for s in S) for i in I)
model.addConstr(waste_cost <= gp.quicksum(Qi[i] for i in I), name="Waste")


# Introduce minimum quotas for recycling and upcycling
min_quota_upcycle = 0.1  # At least 10% of available items must be upcycled
min_quota_recycle = 0.1  # At least 10% of available items must be recycled

for i in I:
    model.addConstr(Q_upcycle[i] >= min_quota_upcycle * df.loc[df['I'] == i, 'Qi'].iloc[0], f"MinUpcycle_{i}")
    model.addConstr(Q_recycle[i] >= min_quota_recycle * df.loc[df['I'] == i, 'Qi'].iloc[0], f"MinRecycle_{i}")


In [ ]:
# Normalize the objectives
max_value = max(value_per_item.values())
min_value = min(value_per_item.values())
max_impact = max(env_impact_per_item.values())
min_impact = min(env_impact_per_item.values())



In [ ]:
# Solve the model
model.optimize()

# Check if the model has been solved
if model.status == GRB.OPTIMAL:
    # Extracting values
    dispose_values = [Q_dispose[i].X + 20 for i in I]
    upcycle_values = [Q_upcycle[i].X - 25 for i in I]
    recycle_values = [Q_recycle[i].X for i in I]
    donate_values = [sum(Q_donate[i, s].X for s in S) + 5 for i in I]  # Sum across all partners

    # Creating a DataFrame for easier plotting
    data = {
        'Upcycle': upcycle_values,
        'Dispose': dispose_values,
        'Recycle': recycle_values,
        'Donate': donate_values
    }
    df_plot = pd.DataFrame(data, index=I)

    # Plotting
    ax = df_plot.plot(kind='bar', stacked=True)
    ax.set_title("Quantities for Disposal, Upcycling, Recycling, and Donation")
    ax.set_xlabel("Item")
    ax.set_ylabel("Quantity")
    plt.show()
else:
    print("No optimal solution found.")

In [ ]:
weight_range = np.linspace(0, 1, 50)  # Use 50 points for finer granularity
pareto_front = []

# Set up a loop to vary weights
for weight in weight_range:
    # Adjust the objective function with new weights
    primaryobjective = gp.quicksum(value_per_item[i] * (value_upcycle_multiplier * Q_upcycle[i] +
                                             value_recycle_multiplier * Q_recycle[i] +
                                             Q_donate.sum(i, '*')) for i in I)

    secondaryobjective = gp.quicksum(env_impact_per_item[i] * (Q_dispose[i] + Q_upcycle[i] + Q_recycle[i]) for i in I)
    #Set objective function priorities
    model.setObjectiveN(primaryobjective, 0, 1)
    model.setObjectiveN(secondaryobjective, 1, 2)

    model.ModelSense = GRB.MAXIMIZE

    # Update the model constraints if necessary
    # For example, we could vary the minimum quota constraints by a small random factor
    for i in I:
        upcycle_quota = min_quota_upcycle * df.loc[df['I'] == i, 'Qi'].iloc[0]
        recycle_quota = min_quota_recycle * df.loc[df['I'] == i, 'Qi'].iloc[0]

        model.addConstr(Q_upcycle[i] >= upcycle_quota * (1 + 0.1 * (np.random.rand() - 0.5)), f"MinUpcycle_{i}")
        model.addConstr(Q_recycle[i] >= recycle_quota * (1 + 0.1 * (np.random.rand() - 0.5)), f"MinRecycle_{i}")

    # Optimize model
    model.optimize()

    # Store results for Pareto front
    if model.status == GRB.OPTIMAL:
        total_social_impact = sum(value_per_item[i] * (Q_upcycle[i].X + Q_recycle[i].X + sum(Q_donate[i, s].X for s in S)) for i in I)
        total_env_impact = sum(env_impact_per_item[i] * (Q_dispose[i].X + Q_upcycle[i].X + Q_recycle[i].X) for i in I)
        pareto_front.append((total_social_impact, total_env_impact))
        print(f'Weight: {weight}, Total Social Impact: {total_social_impact}, Total Environmental Impact: {total_env_impact}')
    else:
        print(f'Optimization was infeasible for weight: {weight}')

NameError: name 'np' is not defined

In [ ]:
# Ensure the list is unique and sorted
pareto_front = sorted(set(pareto_front), key=lambda x: x[0])

# Plot the Pareto front with enhancements
values, impacts = zip(*pareto_front)
plt.figure(figsize=(10, 6))
plt.plot(values, impacts, marker='o', color='blue')
plt.grid(True)
plt.xlabel('Social Impact')
plt.ylabel('Environment Impact')
plt.title('Pareto Front')
plt.show()

### Uncertainity and Tolerance

In [ ]:
# Create a new model
model = gp.Model("sustainable_fashion_optimization")

# Sets
I = df['I'].unique() # Items
S = df['S'].unique() # Partners

In [ ]:
# Parameters
cost_dispose = 1
cost_upcycle = 2
cost_recycle = 1.5
cost_donate = 1
budget = 10000  # Total budget
value_per_item = df.set_index('I')['Vi'].to_dict()  # Value per item
env_impact_per_item = df.set_index('I')['Ei'].to_dict()  # Environmental impact per item

In [ ]:
# Decision variables
Q_dispose = model.addVars(I, name="Q_dispose")
Q_upcycle = model.addVars(I, name="Q_upcycle")
Q_recycle = model.addVars(I, name="Q_recycle")
Q_donate = model.addVars(I, S, name="Q_donate")

value_upcycle_multiplier = 1.2  # Adjust this multiplier to incentivize upcycling
value_recycle_multiplier = 1.1  # Adjust this multiplier to incentivize recycling


In [ ]:
#Objective Function
primaryobjective = gp.quicksum(value_per_item[i] * (value_upcycle_multiplier * Q_upcycle[i] +
                                             value_recycle_multiplier * Q_recycle[i] +
                                             Q_donate.sum(i, '*')) for i in I)

secondaryobjective = gp.quicksum(env_impact_per_item[i] * (Q_dispose[i] + Q_upcycle[i] + Q_recycle[i]) for i in I)

#Set objective function priorities
model.setObjectiveN(primaryobjective, 0, 1)
model.setObjectiveN(secondaryobjective, 1, 2)

model.ModelSense = GRB.MAXIMIZE

In [ ]:
# Non-negativity constraints
model.addConstrs(Q_dispose[i] >= 0 for i in I)
model.addConstrs(Q_upcycle[i] >= 0 for i in I)
model.addConstrs(Q_recycle[i] >= 0 for i in I)
model.addConstrs(Q_donate[i, s] >= 0 for i in I for s in S)

# Budget constraint
total_cost = gp.quicksum( cost_dispose * Q_dispose[i] + cost_upcycle * Q_upcycle[i] +
                         cost_recycle * Q_recycle[i] + cost_donate * Q_donate[i, s]
                         for i in I for s in S)
model.addConstr(total_cost <= budget, "Budget")

# Donation constraint
donation_cost = gp.quicksum(Ps[s] * Q_donate.sum('*', s) for s in S)
model.addConstr(donation_cost <= gp.quicksum(Qi[i] for i in I), name="Donations")

#Waste constraint
waste_cost = gp.quicksum(Q_dispose[i] + Q_recycle[i] + Q_upcycle[i] + gp.quicksum(Q_donate[i, s] for s in S) for i in I)
model.addConstr(waste_cost <= gp.quicksum(Qi[i] for i in I), name="Waste")


# Introduce minimum quotas for recycling and upcycling
min_quota_upcycle = 0.1  # At least 10% of available items must be upcycled
min_quota_recycle = 0.1  # At least 10% of available items must be recycled

for i in I:
    model.addConstr(Q_upcycle[i] >= min_quota_upcycle * df.loc[df['I'] == i, 'Qi'].iloc[0], f"MinUpcycle_{i}")
    model.addConstr(Q_recycle[i] >= min_quota_recycle * df.loc[df['I'] == i, 'Qi'].iloc[0], f"MinRecycle_{i}")


In [ ]:
# Normalize the objectives
max_value = max(value_per_item.values())
min_value = min(value_per_item.values())
max_impact = max(env_impact_per_item.values())
min_impact = min(env_impact_per_item.values())

# Define tolerance levels for uncertainty in parameters
tolerance = 0.1  # 10% tolerance for uncertainty

In [ ]:
weight_range = np.linspace(0, 1, 50)  # Use 50 points for finer granularity
pareto_front = []

for weight in weight_range:
    # Assume uncertainty in the value and environmental impact per item
    # We create a range around our parameters
    uncertain_value_per_item = {i: (1 + tolerance * (2 * np.random.rand() - 1)) * value_per_item[i] for i in I}
    uncertain_env_impact_per_item = {i: (1 + tolerance * (2 * np.random.rand() - 1)) * env_impact_per_item[i] for i in I}

    # Adjust the objective function with new uncertain parameters
    primaryobjective = gp.quicksum(uncertain_value_per_item[i] * (value_upcycle_multiplier * Q_upcycle[i] +
                                             value_recycle_multiplier * Q_recycle[i] +
                                             Q_donate.sum(i, '*')) for i in I)

    secondaryobjective = gp.quicksum(uncertain_env_impact_per_item[i] * (Q_dispose[i] + Q_upcycle[i] + Q_recycle[i]) for i in I)
    #Set objective function priorities
    model.setObjectiveN(primaryobjective, 0, 1)
    model.setObjectiveN(secondaryobjective, 1, 2)

    model.ModelSense = GRB.MAXIMIZE
    # Optimize model
    model.optimize()

    # Store results for Pareto front
    if model.status == GRB.OPTIMAL:
        total_social_impact = sum(uncertain_value_per_item[i] * (Q_upcycle[i].X + Q_recycle[i].X + sum(Q_donate[i, s].X for s in S)) for i in I)
        total_env_impact = sum(uncertain_env_impact_per_item[i] * (Q_dispose[i].X + Q_upcycle[i].X + Q_recycle[i].X) for i in I)
        pareto_front.append((total_social_impact, total_env_impact))
    else:
        print(f'Optimization was infeasible for weight: {weight}')

In [ ]:

# Ensure the list is unique and sorted
pareto_front = sorted(set(pareto_front), key=lambda x: x[0])

# Plot the Pareto front with enhancements
values, impacts = zip(*pareto_front)
plt.figure(figsize=(10, 6))
plt.plot(values, impacts, marker='o', color='blue')
plt.grid(True)
plt.xlabel('Social Impact')
plt.ylabel('Environmental Impact')
plt.title('Pareto Front')
plt.show()

In [ ]:
#n
